# A1.13 · Resource overload

**Function A — Securing AI Architectures → CyberTravels' Architecture, and Every Risk It Carries**  ·  *Security of AI*

Builds on **[A1.12 · Cascading hallucination](https://spbreed.github.io/cyber-commons/lessons/A1.12.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Run a loop with no ceiling and count what it consumes before anything notices.

**Why a security engineer needs it.** An agent consumes budget, tokens, API quota or downstream capacity without bound, and the failure is denial of service against your own systems. The control it builds is: budgets and stop conditions bound to the loop (A3.4).

This is a **risk** lesson: it shows the failure happening before anything tries to stop it, so the control that follows is answering something you have already watched go wrong.

## 1 · The hook

The loop had no ceiling, so it ran until something outside it stopped the run. That something was the invoice. In a different configuration it is a rate limit on a system you do not own, which is somebody else's outage.

> **At CyberTravels.** A booking loop with no ceiling runs until something outside it stops the run. At CyberTravels that something is either the travel API's rate limit — somebody else's outage — or the invoice.

## 2 · The framework

```
   plan --> act --> observe --> plan --> act --> observe --> ...
     ^                                                        |
     +--------------------------------------------------------+

   no token ceiling . no step ceiling . no wall clock . no spend cap
   -> the stop condition is external: an invoice, a rate limit, a person
```

**OWASP T4 — Resource Overload. LLM10 — Unbounded Consumption.**

The **agent_runtime** loops: plan, act, observe, decide again. The loop is the
component that makes an agent an agent, and a loop with no exit condition runs
until something outside it intervenes.

What intervenes, in practice, is a bill, a rate limit, or a person at 3am.

Four resources drain, and they fail differently:

**Tokens and money** — the visible one, discovered on an invoice.

**Downstream capacity** — the one that hurts other people. An agent retrying a
failing API in a tight loop is a denial-of-service attack on your own service,
launched from inside your perimeter by something with valid credentials.

**Rate limit budget** — shared with the humans who need it. The agent exhausts
the quota and the on-call engineer cannot query the API they need.

**Wall-clock time in a critical path** — a workflow step that never returns.

This is a security risk rather than a cost problem for two reasons. It is
**reachable by an attacker**: a task that cannot succeed is easy to construct
via A1.3, and costs the attacker nothing. And it is **availability**, which is
one third of the triad regardless of how the outage was caused.

> **Where this lands on the reference architecture.**
>
> ```
> ingress -> orchestrator -> agent_runtime -> model
>                                |              |
>                          messaging        tools / mcp
>                                |              |
>                       knowledge / memory   egress
>            identity + policy wrap every call · observability records it
> ```

## 3 · The risk, realised

A task that cannot succeed, and a loop with no ceiling.

## 4 · The check, as a skill

CyberTravels' agent does not know the task is impossible. The skill gives it one, measures the three costs, and reports the one that lands on somebody else: the downstream capacity its retries consumed.

In [ ]:
# skills/threats/unbounded-loop-cost-probe/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: unbounded-loop-cost-probe
description: >-
  Give an agent a task it cannot complete and measure what it spends and who
  else pays — tokens, wall time, and the downstream capacity consumed by its
  retries. Use when reviewing loop termination, retry policy, budgets, or a
  scheduled agent nobody watches.
allowed-tools: Read, Grep, Glob
---

# The agent does not know the task is impossible

Resource overload rarely needs an attacker. It needs a task with no completion
condition and a loop with no stop condition, and the cost lands in two places:
your bill, and a downstream service's capacity — where the rejections hit
whoever else was using it.

## When to use this

Before running any agent unattended or on a schedule, and after adding a retry.

## Procedure

**1 — Find the stop conditions.** Step ceiling, token budget, wall-clock
deadline, cost ceiling, and a condition that recognises "this cannot be done".
Record which exist. A loop whose only exit is success has no exit.

**2 — Construct an impossible-but-plausible task.** Not malformed — plausible.
A query against data that does not exist, a fix for a test that cannot pass. The
agent must believe it is making progress.

**3 — Run it with instrumentation and a hard external kill.** The kill is the
safety net; if it is the thing that stops the run, that is the result.

**4 — Record the three costs.** Tokens and money; wall-clock; and downstream
calls — with the rejection rate the downstream started returning. The third is
the one that turns your incident into somebody else's.

**5 — Set the budget from the measurement.** A ceiling chosen from an observed
distribution is defensible; one chosen from a round number is a guess. State
what a legitimate run costs at p95 and set the ceiling above that.

## Output contract

```json
{
  "stop_conditions": {"steps": false, "tokens": false, "wallclock": false, "cost": false, "impossibility": false},
  "run": {"stopped_by": "condition|external_kill", "steps": 0, "tokens": 0, "seconds": 0},
  "downstream": {"calls": 0, "rejections": 0, "affected_others": true},
  "recommended_budget": {"basis": "p95 of legitimate runs", "steps": 0, "tokens": 0}
}
```

## Failure modes

- **Using a malformed task.** The agent gives up, and you learn nothing.
- **Counting only tokens.** The downstream capacity is the externality.
- **Setting a round-number ceiling.** Measure first, or the budget either
  breaks legitimate runs or never fires.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/threats/unbounded-loop-cost-probe/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/threats/unbounded-loop-cost-probe/scripts/unbounded_loop_cost_probe.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Run an agent at an impossible task and measure what it spends, and who else pays for it.

This is the executable half of the `unbounded-loop-cost-probe` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

DOWNSTREAM = {"calls": 0, "capacity": 50, "rejected": 0}

def flaky_api(query):
    """A downstream service. It is not broken - the query cannot be satisfied."""
    DOWNSTREAM["calls"] += 1
    if DOWNSTREAM["calls"] > DOWNSTREAM["capacity"]:
        DOWNSTREAM["rejected"] += 1
        return {"error": "capacity exceeded"}
    return {"result": None}                     # no match, ever

def agent_loop(task, max_steps=None):
    """plan -> act -> observe -> decide again. Stops when it succeeds."""
    steps, tokens = 0, 0
    while True:
        steps += 1
        tokens += 1800
        result = flaky_api(task)
        if result.get("result"):
            return {"done": True, "steps": steps, "tokens": tokens}
        if max_steps and steps >= max_steps:
            return {"done": False, "steps": steps, "tokens": tokens, "stopped_by": "budget"}
        if steps > 500:                          # the notebook's own safety net
            return {"done": False, "steps": steps, "tokens": tokens, "stopped_by": "runaway"}

r = agent_loop("find the order for customer 99999")     # this order does not exist
print(f"steps taken           : {r['steps']}")
print(f"tokens spent          : {r['tokens']:,}  (about ${r['tokens']/1000*0.002:,.2f})")
print(f"downstream calls      : {DOWNSTREAM['calls']}")
print(f"downstream rejections : {DOWNSTREAM['rejected']}  <- other callers got these")
print(f"stopped by            : {r['stopped_by']}")
print()
print("The agent was not attacked and nothing malfunctioned. It was given a")
print("task that cannot succeed, and the loop did what loops do.")
print()
print(f"{DOWNSTREAM['rejected']} rejections went to whoever else was using that")
print("service - a denial of service launched from inside the perimeter, by")
print("something holding valid credentials.")
assert DOWNSTREAM["rejected"] > 0

## What you just proved

An agent given an impossible task loops until the notebook's own safety net stops it, spending hundreds of thousands of tokens and exhausting a downstream service's capacity — with the rejections landing on whoever else was using that service.

## Your turn

Find the ceiling on one agent loop you run. If there is a token budget but no cap on downstream calls, the cost is bounded and the availability risk is not.

---

**Next → [A1.14 · Repudiation and untraceability](https://spbreed.github.io/cyber-commons/lessons/A1.14.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A1.13.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A1.13.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*